# Generate Enhanced Queries: Qwen3-8B

**Model:** Qwen/Qwen3-8B

**Hardware:** A100 GPU (40GB VRAM)

**Quantization:** None (FP16)

**Temperature:** 0.1 (more focused)

**Batch Size:** 16 (optimized for A100)

**Note:** Qwen3 has thinking mode - we'll disable it to get clean pseudo-documents

---

## Setup

### Step 1: Clone Repository and Install Dependencies

In [1]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21 (required for Pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch
!pip install -q datasets accelerate bitsandbytes

print("\n" + "="*60)
print("Installation complete")
print("="*60)
print("IMPORTANT: Restart runtime now!")
print("   1. Click 'Runtime' -> 'Restart runtime'")
print("   2. Then run cells starting from 'Step 2' below")
print("="*60)

Cloning into 'graduation'...
remote: Enumerating objects: 552, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 552 (delta 31), reused 73 (delta 23), pack-reused 468 (from 1)
Receiving objects: 100% (552/552), 20.54 MiB | 24.21 MiB/s, done.
Resolving deltas: 100% (203/203), done.
/content/graduation/arabic-rag-query-enhancement
Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /usr/lib/

### Step 2: Mount Google Drive and Configure Environment (Run After Restart)

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project
%cd /content/graduation/arabic-rag-query-enhancement

# Configure environment
import os
import sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Add src to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

print("\nEnvironment configured")
print("Ready to run experiment")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/graduation/arabic-rag-query-enhancement
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)

Environment configured
Ready to run experiment


## Import Modules

In [2]:
from src.utils.data_loader import MIRACLDataLoader
from src.enhancers.query2doc import Query2DocEnhancer

import torch
from tqdm.notebook import tqdm
import re

print("Modules imported")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Modules imported
GPU Available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 39.5 GB


## Load Data

In [3]:
# Load MIRACL Arabic dev set
data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

print(f"\nDataset Statistics:")
print(f"  Queries: {len(topics)}")
print(f"  Qrels: {len(qrels)}")

# Show sample
sample_qid = list(topics.keys())[0]
print(f"\nSample Query:")
print(f"  ID: {sample_qid}")
print(f"  Text: {topics[sample_qid]['title']}")
print(f"  Relevant docs: {len(qrels.get(sample_qid, {}))}")

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries

Dataset Statistics:
  Queries: 2896
  Qrels: 2896

Sample Query:
  ID: 8099
  Text: من هو علي بن محمد السمري؟
  Relevant docs: 10


## Initialize Qwen3-8B Enhancer

In [8]:
print("Initializing Qwen3-8B enhancer...")
print("Model: Qwen/Qwen3-8B")
print("Quantization: None (FP16)")
print("Temperature: 0.1 (more focused)")
print("Batch size: 16 (optimized for A100)")
print("\nThis will download ~16GB on first run...\n")

# Reload module to ensure clean state
import importlib
import sys
if 'src.enhancers.query2doc' in sys.modules:
    del sys.modules['src.enhancers.query2doc']
from src.enhancers.query2doc import Query2DocEnhancer

enhancer = Query2DocEnhancer(
    model_name="Qwen/Qwen3-8B",
    max_new_tokens=128,
    temperature=0.1,  # Lower temperature for more focused generation
    top_p=0.9,
    batch_size=16  # Larger batch for A100
)

print("\nQwen3-8B enhancer ready")
print(f"Model device: {enhancer.model.device}")
print(f"Batch size: 16")

Initializing Qwen3-8B enhancer...
Model: Qwen/Qwen3-8B
Quantization: None (FP16)
Temperature: 0.1 (more focused)
Batch size: 16 (optimized for A100)

This will download ~16GB on first run...

Loading Qwen/Qwen3-8B in float16...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

✓ Model loaded on cuda:0
✓ Batch size: 16 (processing 16 queries at once)
✓ Max tokens: 128 (shorter = faster)

Qwen3-8B enhancer ready
Model device: cuda:0
Batch size: 16


## Patch Enhancer to Strip Thinking Tags

Qwen3 has a thinking mode that produces `<think>...</think>` tags. We need to strip these.

In [15]:
import types
import re

# Get the original enhance method from the class
_original_enhance = enhancer.__class__.enhance
_original_enhance_batch_parallel = enhancer.__class__.enhance_batch_parallel

def strip_thinking_tags(text):
    """Remove all thinking tags (including empty ones)"""
    # Remove <think>...</think> blocks
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    # Clean up extra whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def enhance_no_thinking(self, query, query_id=None):
    """Enhanced method that disables thinking mode and strips tags"""
    # Format prompt with /no_think directive
    messages = [
        {"role": "system", "content": self.system_prompt + "\n/no_think"},
        {"role": "user", "content": query}
    ]
    
    text = self.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Generate pseudo-document
    model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
    
    with torch.no_grad():
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p=self.top_p,
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id
        )
    
    # Decode response
    generated_ids = [
        output_ids[len(input_ids):] 
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    pseudo_doc = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # Strip any remaining thinking tags
    pseudo_doc = strip_thinking_tags(pseudo_doc)
    
    # Combine: original query + pseudo-document
    enhanced = f"{query} {pseudo_doc}"
    return enhanced

def enhance_batch_parallel_no_thinking(self, queries, query_ids=None):
    """Enhanced batch method that disables thinking mode and strips tags"""
    # Format all prompts with /no_think directive
    all_messages = [
        [
            {"role": "system", "content": self.system_prompt + "\n/no_think"},
            {"role": "user", "content": query}
        ]
        for query in queries
    ]
    
    # Apply chat template to all
    texts = [
        self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        for messages in all_messages
    ]
    
    # Tokenize with padding
    model_inputs = self.tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(self.model.device)
    
    # Generate for all queries at once
    with torch.no_grad():
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p=self.top_p,
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id
        )
    
    # Decode all responses
    input_lengths = model_inputs.input_ids.shape[1]
    pseudo_docs = self.tokenizer.batch_decode(
        generated_ids[:, input_lengths:],
        skip_special_tokens=True
    )
    
    # Strip thinking tags from all pseudo-documents
    pseudo_docs = [strip_thinking_tags(doc) for doc in pseudo_docs]
    
    # Combine original queries with pseudo-documents
    enhanced = [f"{q} {doc}" for q, doc in zip(queries, pseudo_docs)]
    return enhanced

# Patch the instance
enhancer.enhance = types.MethodType(enhance_no_thinking, enhancer)
enhancer.enhance_batch_parallel = types.MethodType(enhance_batch_parallel_no_thinking, enhancer)

print("✓ /no_think directive + tag stripping enabled")
print("All thinking tags will be removed")


✓ /no_think directive + tag stripping enabled
All thinking tags will be removed


## Check GPU Memory

In [6]:
if torch.cuda.is_available():
    print("=== GPU Status ===")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"Memory total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3:.2f} GB")

=== GPU Status ===
GPU: NVIDIA A100-SXM4-40GB
Memory allocated: 15.26 GB
Memory reserved: 15.26 GB
Memory total: 39.49 GB
Memory free: 24.24 GB


## Test on Sample Query

In [17]:
# Test enhancer on sample query
sample_query = topics[sample_qid]['title']
print(f"Testing enhancer on sample query...\n")
print(f"Original: {sample_query}")
print(f"\nGenerating pseudo-document (with thinking tag removal)...")

enhanced_sample = enhancer.enhance(sample_query)
print(f"\nEnhanced: {enhanced_sample[:5000]}...")
print(f"\nLength: {len(sample_query)} -> {len(enhanced_sample)} chars")
print(f"Expansion ratio: {len(enhanced_sample)/len(sample_query):.2f}x")

# Check if thinking tags were present
if '<think>' in enhanced_sample or '</think>' in enhanced_sample:
    print("\nWARNING: Thinking tags still present! Check stripping logic.")
else:
    print("\n✓ Thinking tags successfully removed.")


Testing enhancer on sample query...

Original: من هو علي بن محمد السمري؟

Generating pseudo-document (with thinking tag removal)...

Enhanced: من هو علي بن محمد السمري؟ علي بن محمد السمري هو شخصية بارزة في مجال الترجمة والنشر في المملكة العربية السعودية. وهو مترجم وكاتب ومحقق للكتب، يُعرف بمساهماته في نشر الأعمال الأدبية والعلمية باللغة العربية. كما أنه يُعتبر من أبرز المترجمين الذين ساهموا في ترجمة العديد من الكتب الأجنبية إلى اللغة العربية، مما ساعد في إثراء المعرفة الثقافية والعلمية في المجتمع العربي. يُقدّر عمله في مجال الترجمة والنشر...

Length: 25 -> 401 chars
Expansion ratio: 16.04x

✓ Thinking tags successfully removed.


## Generate Enhanced Queries for All Data

In [18]:
import time

print("="*60)
print("GENERATING ENHANCED QUERIES: Qwen3-8B")
print("="*60)

# Prepare queries
query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nTotal queries: {len(query_texts)}")
print(f"Batch size: 16")
print(f"Expected batches: {len(query_texts) // 16 + 1}")
print(f"Expected time: ~20-30 minutes\n")

start_time = time.time()

# Apply Query2Doc enhancement
enhanced_queries = enhancer.enhance_batch(
    query_texts,
    query_ids,
    show_progress=True
)

elapsed = time.time() - start_time

print(f"\nEnhanced {len(enhanced_queries)} queries")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Queries per minute: {len(query_texts)/(elapsed/60):.1f}")

GENERATING ENHANCED QUERIES: Qwen3-8B

Total queries: 2896
Batch size: 16
Expected batches: 182
Expected time: ~20-30 minutes



Enhancing batches: 100%|██████████| 181/181 [24:29<00:00,  8.12s/it]


Enhanced 2896 queries
Total time: 24.5 minutes
Queries per minute: 118.2


## Verify No Thinking Tags in Output

In [19]:
# Check if any thinking tags remain
thinking_tags_found = sum(1 for eq in enhanced_queries if '<think>' in eq or '</think>' in eq)

if thinking_tags_found > 0:
    print(f"WARNING: Found thinking tags in {thinking_tags_found} queries!")
    print("Cleaning them now...")
    enhanced_queries = [strip_thinking_tags(eq) for eq in enhanced_queries]
    print("Cleaned.")
else:
    print("No thinking tags found in output.")

No thinking tags found in output.


## Show Enhancement Examples

In [20]:
print("\nEnhancement Examples:\n")
for i in range(min(5, len(query_texts))):
    print(f"Query {i+1}:")
    print(f"  Original ({len(query_texts[i])} chars): {query_texts[i]}")
    print(f"  Enhanced ({len(enhanced_queries[i])} chars): {enhanced_queries[i][:200]}...")  # First 200 chars
    print(f"  Expansion: {len(enhanced_queries[i])/len(query_texts[i]):.2f}x")
    print()


Enhancement Examples:

Query 1:
  Original (25 chars): من هو علي بن محمد السمري؟
  Enhanced (330 chars): من هو علي بن محمد السمري؟ علي بن محمد السمري هو شخصية بارزة في مجال الترجمة والنشر في المملكة العربية السعودية. وهو مترجم وكاتب ومحقق للكتب، يُعرف بمساهماته في نشر الأعمال الأدبية والعلمية باللغة العر...
  Expansion: 13.20x

Query 2:
  Original (34 chars): متى تم إستخدام الغوّاصات لأول مرة؟
  Enhanced (347 chars): متى تم إستخدام الغوّاصات لأول مرة؟ تم استخدام الغواصات لأول مرة في القرن الثامن عشر، حيث أُطلق أول نموذج للغواصة بواسطة توماس ديفيدسون في عام 1776. لكن الغواصة الحديثة التي تستخدم الأسطوانات الهوائية ...
  Expansion: 10.21x

Query 3:
  Original (28 chars): من هو القديس المسمى بالصخرة؟
  Enhanced (317 chars): من هو القديس المسمى بالصخرة؟ القديس المسمى بالصخرة هو القديس بطرس، وهو أحد الرسل الأربعة في المسيحية. يُعتبر بطرس من أعظم الرسل وأكثرهم تأثيرًا، حيث كان من أول من سمع خطاب المسيح واتبعه. يُعتقد أن اسم...
  Expansion: 11.32x

Query 4:
  Original (33 chars): هل يرتبط ال

## Query Expansion Statistics

In [21]:
import numpy as np

# Calculate statistics
original_lengths = [len(q) for q in query_texts]
enhanced_lengths = [len(eq) for eq in enhanced_queries]
expansion_ratios = [e/o if o > 0 else 0 for o, e in zip(original_lengths, enhanced_lengths)]

print("=== Query Expansion Statistics ===")
print(f"\nOriginal queries:")
print(f"  Mean length: {np.mean(original_lengths):.1f} chars")
print(f"  Median length: {np.median(original_lengths):.1f} chars")
print(f"  Min/Max: {min(original_lengths)} / {max(original_lengths)} chars")

print(f"\nEnhanced queries:")
print(f"  Mean length: {np.mean(enhanced_lengths):.1f} chars")
print(f"  Median length: {np.median(enhanced_lengths):.1f} chars")
print(f"  Min/Max: {min(enhanced_lengths)} / {max(enhanced_lengths)} chars")

print(f"\nExpansion ratio:")
print(f"  Mean: {np.mean(expansion_ratios):.2f}x")
print(f"  Median: {np.median(expansion_ratios):.2f}x")
print(f"  Min/Max: {min(expansion_ratios):.2f}x / {max(expansion_ratios):.2f}x")

=== Query Expansion Statistics ===

Original queries:
  Mean length: 29.5 chars
  Median length: 27.0 chars
  Min/Max: 12 / 101 chars

Enhanced queries:
  Mean length: 273.3 chars
  Median length: 312.0 chars
  Min/Max: 34 / 509 chars

Expansion ratio:
  Mean: 10.61x
  Median: 10.20x
  Min/Max: 1.84x / 29.50x


## Save Enhanced Queries

In [23]:
import pickle

# Save enhanced queries
output_file = 'enhanced_queries_qwen3_8b.pkl'

with open(output_file, 'wb') as f:
    pickle.dump({
        'query_ids': query_ids,
        'original': query_texts,
        'enhanced': enhanced_queries,
        'model': 'Qwen/Qwen3-8B',
        'config': {
            'max_new_tokens': 128,
            'temperature': 0.1,
            'top_p': 0.9,
            'batch_size': 16,
            'quantization': 'None (FP16)',
            'hardware': 'A100 GPU',
            'thinking_tags_stripped': True
        },
        'stats': {
            'total_queries': len(query_texts),
            'mean_original_length': np.mean(original_lengths),
            'mean_enhanced_length': np.mean(enhanced_lengths),
            'mean_expansion_ratio': np.mean(expansion_ratios),
            'generation_time_minutes': elapsed/60
        }
    }, f)

print(f"Enhanced queries saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024**2:.1f} MB")

# Also save to Google Drive
drive_path = '/content/enhanced_queries_qwen3_8b.pkl'
!cp {output_file} {drive_path}
print(f"\nBackup saved to Google Drive: {drive_path}")

Enhanced queries saved to: enhanced_queries_qwen3_8b.pkl
File size: 1.5 MB

Backup saved to Google Drive: /content/enhanced_queries_qwen3_8b.pkl


## Summary

In [ ]:
print("="*60)
print("GENERATION COMPLETE")
print("="*60)
print(f"\nModel: Qwen3-8B")
print(f"Quantization: None (FP16)")
print(f"Temperature: 0.1")
print(f"Batch size: 16")
print(f"Hardware: {torch.cuda.get_device_name(0)}")
print(f"Thinking tags stripped: Yes")
print(f"\nQueries processed: {len(enhanced_queries)}")
print(f"Generation time: {elapsed/60:.1f} minutes")
print(f"Average expansion: {np.mean(expansion_ratios):.2f}x")
print(f"\nOutput file: {output_file}")
print(f"\nNext step: Use evaluate_enhanced_queries.ipynb to test with Dense and BM25")